In [2]:
# Importing necessary libraries for EDA
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn



In [4]:
# Step 1: Load and Understand the Dataset

import pandas as pd

# Load the dataset (replace 'your_file.csv' with the actual file path)
df = pd.read_csv('data/data.csv')

# Inspect the structure of the dataset
print("Dataset Shape:", df.shape)
print("\nDataset Info:")
df.info()

# Display summary statistics
print("\nSummary Statistics:")
print(df.describe())

# Display first few rows of the dataset
print("\nFirst Few Rows:")
print(df.head())


Dataset Shape: (95662, 16)

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 95662 entries, 0 to 95661
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   TransactionId         95662 non-null  object 
 1   BatchId               95662 non-null  object 
 2   AccountId             95662 non-null  object 
 3   SubscriptionId        95662 non-null  object 
 4   CustomerId            95662 non-null  object 
 5   CurrencyCode          95662 non-null  object 
 6   CountryCode           95662 non-null  int64  
 7   ProviderId            95662 non-null  object 
 8   ProductId             95662 non-null  object 
 9   ProductCategory       95662 non-null  object 
 10  ChannelId             95662 non-null  object 
 11  Amount                95662 non-null  float64
 12  Value                 95662 non-null  int64  
 13  TransactionStartTime  95662 non-null  object 
 14  PricingStrategy       95662 

In [5]:
# Step 2: Create Aggregate Features

# Total Transaction Amount per Customer
df['TotalTransactionAmount'] = df.groupby('CustomerId')['Amount'].transform('sum')

# Average Transaction Amount per Customer
df['AverageTransactionAmount'] = df.groupby('CustomerId')['Amount'].transform('mean')

# Transaction Count per Customer
df['TransactionCount'] = df.groupby('CustomerId')['TransactionId'].transform('count')

# Standard Deviation of Transaction Amounts per Customer
df['StdTransactionAmount'] = df.groupby('CustomerId')['Amount'].transform('std')

# Verify the new features
print("Aggregate Features Preview:")
print(df[['CustomerId', 'TotalTransactionAmount', 'AverageTransactionAmount', 'TransactionCount', 'StdTransactionAmount']].head())


Aggregate Features Preview:
        CustomerId  TotalTransactionAmount  AverageTransactionAmount  \
0  CustomerId_4406               109921.75                923.712185   
1  CustomerId_4406               109921.75                923.712185   
2  CustomerId_4683                 1000.00                500.000000   
3   CustomerId_988               228727.20               6019.136842   
4   CustomerId_988               228727.20               6019.136842   

   TransactionCount  StdTransactionAmount  
0               119           3042.294251  
1               119           3042.294251  
2                 2              0.000000  
3                38          17169.241610  
4                38          17169.241610  


In [6]:
# Step 2: Extract Time-Based Features

# Convert TransactionStartTime to datetime format
df['TransactionStartTime'] = pd.to_datetime(df['TransactionStartTime'])

# Extract Transaction Hour
df['TransactionHour'] = df['TransactionStartTime'].dt.hour

# Extract Transaction Day
df['TransactionDay'] = df['TransactionStartTime'].dt.day

# Extract Transaction Month
df['TransactionMonth'] = df['TransactionStartTime'].dt.month

# Extract Transaction Year
df['TransactionYear'] = df['TransactionStartTime'].dt.year

# Verify the new features
print("Time-Based Features Preview:")
print(df[['TransactionStartTime', 'TransactionHour', 'TransactionDay', 'TransactionMonth', 'TransactionYear']].head())


Time-Based Features Preview:
       TransactionStartTime  TransactionHour  TransactionDay  \
0 2018-11-15 02:18:49+00:00                2              15   
1 2018-11-15 02:19:08+00:00                2              15   
2 2018-11-15 02:44:21+00:00                2              15   
3 2018-11-15 03:32:55+00:00                3              15   
4 2018-11-15 03:34:21+00:00                3              15   

   TransactionMonth  TransactionYear  
0                11             2018  
1                11             2018  
2                11             2018  
3                11             2018  
4                11             2018  


In [27]:
# Importing necessary module
import category_encoders as ce

# Define categorical columns for WOE transformation
categorical_columns = ['ProductCategory', 'ProviderId', 'ChannelId']

# Create WOE encoder
woe_encoder = ce.WOEEncoder(cols=categorical_columns)

In [35]:
# Importing necessary module
import category_encoders as ce

# Define categorical columns for WOE transformation
categorical_columns = ['ProviderId']  # Only ProviderId needs WOE encoding

# Initialize WOEEncoder
woe_encoder = ce.WOEEncoder(cols=categorical_columns)

# Fit and transform WOE encoder on data
woe_encoded = woe_encoder.fit_transform(df[categorical_columns], df['FraudResult'])

# Rename WOE encoded columns
woe_encoded.columns = [f"{col}_WOE" for col in categorical_columns]

# Drop original categorical columns and merge WOE encoded columns
df = df.drop(columns=categorical_columns).reset_index(drop=True)
df = pd.concat([df, woe_encoded], axis=1)

# Preview the dataset after WOE encoding
print("WOE Encoded Features Preview:")
print(df.head())

WOE Encoded Features Preview:
         TransactionId         BatchId       AccountId       SubscriptionId  \
0  TransactionId_76871   BatchId_36123  AccountId_3957   SubscriptionId_887   
1  TransactionId_73770   BatchId_15642  AccountId_4841  SubscriptionId_3829   
2  TransactionId_26203   BatchId_53941  AccountId_4229   SubscriptionId_222   
3    TransactionId_380  BatchId_102363   AccountId_648  SubscriptionId_2185   
4  TransactionId_28195   BatchId_38780  AccountId_4841  SubscriptionId_3829   

        CustomerId  CountryCode     ProductId   Amount  Value  \
0  CustomerId_4406          256  ProductId_10   1000.0   1000   
1  CustomerId_4406          256   ProductId_6    -20.0     20   
2  CustomerId_4683          256   ProductId_1    500.0    500   
3   CustomerId_988          256  ProductId_21  20000.0  21800   
4   CustomerId_988          256   ProductId_6   -644.0    644   

       TransactionStartTime  ...  ProductCategory_movies  \
0 2018-11-15 02:18:49+00:00  ...            

In [36]:
# Selecting the relevant WOE-encoded features and other necessary columns for the model
columns_to_use = [
    'Amount', 'Value', 'TotalTransactionAmount', 'AverageTransactionAmount',
    'TransactionCount', 'StdTransactionAmount', 'TransactionHour', 'TransactionDay',
    'TransactionMonth', 'TransactionYear', 'ProviderId_WOE'
]

# Creating a new dataframe for model input
model_data = df[columns_to_use]

# Displaying the prepared data for modeling
print("Model Data Preview:")
print(model_data.head())


Model Data Preview:
    Amount  Value  TotalTransactionAmount  AverageTransactionAmount  \
0   1000.0   1000               109921.75                923.712185   
1    -20.0     20               109921.75                923.712185   
2    500.0    500                 1000.00                500.000000   
3  20000.0  21800               228727.20               6019.136842   
4   -644.0    644               228727.20               6019.136842   

   TransactionCount  StdTransactionAmount  TransactionHour  TransactionDay  \
0               119           3042.294251                2              15   
1               119           3042.294251                2              15   
2                 2              0.000000                2              15   
3                38          17169.241610                3              15   
4                38          17169.241610                3              15   

   TransactionMonth  TransactionYear  ProviderId_WOE  
0                11          

In [9]:
# Step 5: Normalize/Standardize Numerical Features

from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Select numerical columns
numerical_columns = ['TotalTransactionAmount', 'AverageTransactionAmount', 
                     'TransactionCount', 'StdTransactionAmount', 'Amount', 'Value']

# Create scalers
normalizer = MinMaxScaler()
standardizer = StandardScaler()

# Normalize features (range [0, 1])
df_normalized = df.copy()
df_normalized[numerical_columns] = normalizer.fit_transform(df[numerical_columns])

# Standardize features (mean 0, std 1)
df_standardized = df.copy()
df_standardized[numerical_columns] = standardizer.fit_transform(df[numerical_columns])

# Preview normalized and standardized datasets
print("Normalized Features Preview:\n", df_normalized[numerical_columns].head())
print("\nStandardized Features Preview:\n", df_standardized[numerical_columns].head())


Normalized Features Preview:
    TotalTransactionAmount  AverageTransactionAmount  TransactionCount  \
0                0.557522                  0.047184          0.028851   
1                0.557522                  0.047184          0.028851   
2                0.556944                  0.047137          0.000244   
3                0.558153                  0.047749          0.009046   
4                0.558153                  0.047749          0.009046   

   StdTransactionAmount    Amount     Value  
0              0.000919  0.092004  0.000101  
1              0.000919  0.091910  0.000002  
2              0.000000  0.091958  0.000050  
3              0.005187  0.093750  0.002206  
4              0.005187  0.091853  0.000065  

Standardized Features Preview:
    TotalTransactionAmount  AverageTransactionAmount  TransactionCount  \
0                0.170118                 -0.067623         -0.311831   
1                0.170118                 -0.067623         -0.311831   
2  